<a href="https://colab.research.google.com/github/soo423/Python-archive/blob/main/selenium.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Hello")

Hello


In [ ]:
%pip install selenium webdriver-manager beautifulsoup4 pandas openpyxl -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from selenium import webdriver #브라우저를 실행하고 제어
from selenium.webdriver.chrome.service import Service # 크롬 드라이버 실행 파일의 경로와 실행 방식을 관리
from selenium.webdriver.chrome.options import Options #브라우저 옵션(설정)을 지정
from selenium.webdriver.common.by import By #html 요소를 찾는 방법을 지정 by.ID / by.CSS_SELETOR / by.XPATH
from selenium.webdriver.support.ui import WebDriverWait #특정 요소가 나타날 때까지 기다림
from selenium.webdriver.support import expected_conditions as EC #특정 요소가 나타날 때까지 기다리는 조건을 지정
from webdriver_manager.chrome import ChromeDriverManager #크롬 드라이버 자동 설치 및 관리
from bs4 import BeautifulSoup #HTML 파싱 및 데이터 추출
import pandas as pd #데이터 분석 및 조작을 위한 라이브러리
import time #시간 관련 함수 사용 / 지정한 초만큼 실행을 멈추고 대기
import re #정규 표현식 사용

In [ ]:
chrome_options = Options()
chrome_options.add_experimental_option("detach", True)#detach=True : 코드 실행이 끝나도 브라우저가 자동으로 닫히지 않음
chrome_options.add_experimental_option("excludeSwitches", ["enable-logging"]) #크롬 실행 시 출력되는 불필요한 로그를 제거
service = Service(ChromeDriverManager().install())#알맞은 버전의 크롬드라이버를 자동 설치하고 경로를 반환
#반환된 경로로 드라이버 실행 서비스 객체를 만듬
driver = webdriver.Chrome(service=service, options=chrome_options)#실제 크롬 브라우저를 실행
print("드라이버 실행 완료")


드라이버 실행 완료


### 페이지 접속 / 상품 링크 10개 수집

In [ ]:
main_url = "https://www.oliveyoung.co.kr/store/main/getBestList.do?dispCatNo=900000100100001&fltDispCatNo=10000010008&pageIdx=1&rowsPerPage=8"
driver.get(main_url)#브라우저가 해당 url로 이동
#봇 접근 시 캡차를 띄우므로 수동으로 풀 수 있게 300초 간 대기해야됨
wait = WebDriverWait(driver, 300)
wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.best-area")))
time.sleep(2)#요소 감지 후 2초 추가 대기, 페이지 내 다른 요소들까지 완전히 렌더링될 시간
soup = BeautifulSoup(driver.page_source, "html.parser") #현재 브라우저에 렌더링된 html 전체를 문자열로 가져와서 파싱하여 soup 객체를 만듬
items = soup.select("#Container > div.best-area > div.TabsConts.on > ul > li")
sub_url = [
    item.select_one("div.prd_info > a.prd_thumb")["href"]
    for item in items[:10]
]#리스트 컴프리헨션으로 각 상품 카드에서 링크(href)를 뽑아 놓음
print(f"수집된 링크: {len(sub_url)}개")

for i, url in enumerate(sub_url, 1):
    print(f"{i},{url.split('goodsNo=')[1].split("&")[0]}")

수집된 링크: 10개
1,A000000206904
2,A000000238634
3,A000000235485
4,A000000226086
5,A000000219609
6,A000000243330
7,A000000013194
8,A000000247689
9,A000000245459
10,A000000245463


### 상품 상세 페이지 접속 & 상품 정보 수집

In [ ]:
result = [] #수집결과를 담을 빈 리스트

for i, url in enumerate(sub_url, 1):
    print(f"[{i}/10] 크롤링 중...\n", end=" ")

    driver.get(url)#브라우저가 상품 상세 페이지로 이동
    time.sleep(3) #JavaScript가 상품 정보를 렌더링할 시간을 확보함
    soup = BeautifulSoup(driver.page_source, "html.parser")#상세페이지 html을 파싱

    #브랜드명
    try:
        brand = soup.select_one('[class*="btn-brand"]').get_text(strip=True)
    except:
        brand = "없음"

    #상품명
    try:
        name = soup.select_one('[class*="GoodsDetailInfo_title"]').get_text(strip=True)
    except:
        name = "없음"
    #카테고리
    try:
        breacrumb = soup.select_one('[class *="Breadcrumb_breadcrumb-inner"]')
        links = breacrumb.select('a[role="link"]')
        category = ">".join([a.get_text(strip=True) for a in links])
    except:
        category = "없음"
    #정가
    try:
        price = soup.select_one('[data-qa-name="text-product-original-price"]span:first-child').get_text(strip=True)+"원"
    except:
        price = "없음"
    #할인가
    try:
        discount = soup.select_one('[data-qa-name="text-product-discount-price"]span:first-child').get_text(strip=True)+"원"
    except:
        discount = "없음"
    #평점
    try:
        rating_tag = soup.select_one('span.rating') #select_one(): 조건에 맞는 요소를 하나만 반환
        for blind in rating_tag.select('.oyblind'): #select_one() : 조건에 맞는 요소를 모두 찾아 리스트로 반환
            blind.decompose() #decompse()로 제거
        rating = rating_tag.get_text(strip=True)
    except:
        rating = '없음'
    #리뷰건수
    try:
        review_count = soup.select_one('div[class*="review-count"]button span').get_text(strip=True)+"건"
    except:
        review_count = "없음"

    result.append([i, brand, name, category, price, discount, rating, review_count])
    print(f"{brand} / {name[:25]} / 평점 {rating} / 리뷰 {review_count}")

driver.quit() #모든 수집이 종료되면 브라우저를 종료함
print(f"\n 수집 완료 - 총 {len(result)}개")

[1/10] 크롤링 중...
 라로슈포제 / 라로슈포제 시카플라스트 밤 B5+ 100ml  / 평점 4.8 / 리뷰 없음
[2/10] 크롤링 중...
 에스트라 / 에스트라 아토베리어365 크림 80ml 어워즈 / 평점 4.9 / 리뷰 없음
[3/10] 크롤링 중...
 유리아쥬 / [3월 올영픽/품절대란템 NEW 컬러] 유리아 / 평점 4.6 / 리뷰 없음
[4/10] 크롤링 중...
 바이오더마 / 바이오더마 하이드라비오 에센스로션 200ml  / 평점 4.8 / 리뷰 없음
[5/10] 크롤링 중...
 파티온 / [대용량/트러블1등] 파티온 노스카나인 트러블 / 평점 4.9 / 리뷰 없음
[6/10] 크롤링 중...
 리쥬란 / [9중 모공지표 개선/올리브영 단독 런칭] 리 / 평점 4.9 / 리뷰 없음
[7/10] 크롤링 중...
 바이오더마 / 바이오더마 센시비오 H2O 500ml 2입 / 평점 4.9 / 리뷰 없음
[8/10] 크롤링 중...
 라로슈포제 / [3월올영픽/대용량] 라로슈포제 시카밤 B5+ / 평점 4.8 / 리뷰 없음
[9/10] 크롤링 중...
 바이오더마 / 바이오더마 하이드라비오 토너 500ml 기획  / 평점 4.8 / 리뷰 없음
[10/10] 크롤링 중...
 바이오더마 / 바이오더마 시카비오 포마드 100ml 기획 ( / 평점 4.8 / 리뷰 없음

 수집 완료 - 총 10개


### 데이터프레임 확인

In [ ]:
columns = ["순위", "브랜드", "상품명", "카테고리", "정가", "할인가", "평점", "리뷰건수"]
df = pd.DataFrame(result, columns=columns)
df

,순위,브랜드,상품명,카테고리,정가,할인가,평점,리뷰건수
0,1,라로슈포제,라로슈포제 시카플라스트 밤 B5+ 100ml 기획 (+3ml 추가증정),스킨케어>크림>크림,없음,없음,4.8,없음
1,2,에스트라,에스트라 아토베리어365 크림 80ml 어워즈 한정기획 (+30ml+앰플 3ml),스킨케어>크림>크림,없음,없음,4.9,없음
2,3,유리아쥬,[3월 올영픽/품절대란템 NEW 컬러] 유리아쥬 스틱레브르 컬러드 5종,메이크업>립메이크업>립케어,없음,없음,4.6,없음
3,4,바이오더마,바이오더마 하이드라비오 에센스로션 200ml 기획(+안개분사 미스트 증정),스킨케어>에센스/세럼/앰플>에센스/세럼/앰플,없음,없음,4.8,없음
4,5,파티온,[대용량/트러블1등] 파티온 노스카나인 트러블 세럼 50ml 리필 기획(+리필40m...,스킨케어>에센스/세럼/앰플>에센스/세럼/앰플,없음,없음,4.9,없음
5,6,리쥬란,[9중 모공지표 개선/올리브영 단독 런칭] 리쥬란 더마 힐러 포어 타이트닝 겔 마스...,마스크팩>시트팩>겔 마스크,없음,없음,4.9,없음
6,7,바이오더마,바이오더마 센시비오 H2O 500ml 2입,클렌징>워터/밀크>클렌징워터,없음,없음,4.9,없음
7,8,라로슈포제,[3월올영픽/대용량] 라로슈포제 시카밤 B5+ 200ml+시카PRO마스크 루틴 기획...,스킨케어>크림>크림,없음,없음,4.8,없음
8,9,바이오더마,바이오더마 하이드라비오 토너 500ml 기획 (+거품 용기),스킨케어>스킨/토너>스킨/토너,없음,없음,4.8,없음
9,10,바이오더마,바이오더마 시카비오 포마드 100ml 기획 (+거즈 시트 마스크 10매),스킨케어>크림>크림,없음,없음,4.8,없음


In [ ]:
df.to_excel("올리브영_랭킹top10.xlsx", index=False)
print("엑셀 저장 완료 - 올리브영_랭킹top10.xlsx")

엑셀 저장 완료 - 올리브영_랭킹top10.xlsx
